In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]
products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)
stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]
stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)
products_df.show()
stores_df.show()
sales_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
         .when(F.col("order_id") % 7 == 0, "Returned")
         .otherwise("Completed")
    )

    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
         .when(F.col("order_id") % 3 == 1, "Card")
         .otherwise("Cash")
    )

    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )

    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id",
        "left"
    )

    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10, truncate=False)

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+

Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_p

Part A

In [3]:
exercise_1_df = sales_df.select("order_id", "product_id", "quantity", "unit_price")
exercise_1_df.show(10)

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
+--------+----------+--------+----------+
only showing top 10 rows


In [4]:
exercise_2_df = sales_df.filter((F.col("order_status") == "Completed") & (F.col("quantity") > 5))
exercise_2_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|       5|2026-01-06|     S02|       P06|       6|     24000|           5|   Completed|          Cash|         EMP06|
|      23|2026-01-24|     S04|       P06|       8|     24000|           0|   Completed|          Cash|         EMP12|
|      29|2026-01-30|     S02|       P06|       6|     2

In [5]:
exercise_3_df = sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
exercise_3_df.select("order_id", "quantity", "unit_price", "gross_revenue").show(10)

+--------+--------+----------+-------------+
|order_id|quantity|unit_price|gross_revenue|
+--------+--------+----------+-------------+
|       3|       4|     17000|        68000|
|       9|       2|     17000|        34000|
|       2|       3|      3500|        10500|
|       8|       1|      3500|         3500|
|       4|       5|      4500|        22500|
|      10|       3|      4500|        13500|
|       5|       6|     24000|       144000|
|      11|       4|     24000|        96000|
|       1|       2|     14000|        28000|
|       7|       8|     14000|       112000|
+--------+--------+----------+-------------+
only showing top 10 rows


In [6]:
exercise_4_df = sales_df.withColumnRenamed("order_status", "transaction_status").drop("payment_method")
exercise_4_df.show(10)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|       2|2026-01-03|     S01|       P03|       3|      3500|           0|         Completed|         EMP03|
|       8|2026-01-09|     S03|       P03|       1|      3500|           5|         Completed|         EMP09|
|       4|2026-01-05|     S02|       P05|       5|      4500|           0|         Completed|         EMP05|
|      10|2026-01-11|     S04|       P05|       3|      4500|          10|         Completed|         EMP11|
|       5|2026-01-0

Part B

In [7]:
exercise_5_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity_sold")).orderBy(F.desc("total_quantity_sold"))
exercise_5_df.show()

+----------+-------------------+
|product_id|total_quantity_sold|
+----------+-------------------+
|       P02|              83336|
|       P04|              83334|
|       P06|              83330|
|       P01|              66668|
|       P03|              66667|
|       P05|              66665|
+----------+-------------------+



In [8]:
exercise_6_df = (sales_df.join(products_df, "product_id", "left")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("category").agg(F.sum("gross_revenue").alias("total_gross_revenue"))
    .orderBy(F.desc("total_gross_revenue")))
exercise_6_df.show(truncate=False)

+-----------+-------------------+
|category   |total_gross_revenue|
+-----------+-------------------+
|Electronics|7166704000         |
|Office     |1416678000         |
|Networking |299992500          |
|Accessories|233334500          |
+-----------+-------------------+



In [9]:
exercise_7_df = (sales_df.join(stores_df, "store_id", "left").groupBy("city")
    .agg(F.count("*").alias("transaction_count")).orderBy(F.desc("transaction_count")))
exercise_7_df.show()

+---------+-----------------+
|     city|transaction_count|
+---------+-----------------+
|Bengaluru|            25001|
|Hyderabad|            25001|
|   Mumbai|            24999|
|    Delhi|            24999|
+---------+-----------------+



In [10]:
exercise_8_df = (sales_df.filter(F.col("order_status") == "Completed")
    .join(products_df, "product_id", "left").join(stores_df, "store_id", "left")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("product_id", "product_name", "category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))
    .orderBy(F.desc("total_gross_revenue")).limit(5))
exercise_8_df.show(truncate=False)

+----------+------------+-----------+-------------------+
|product_id|product_name|category   |total_gross_revenue|
+----------+------------+-----------+-------------------+
|P01       |Laptop      |Electronics|3116940000         |
|P06       |Tablet      |Electronics|1558416000         |
|P04       |Printer     |Office     |1103912000         |
|P02       |Monitor     |Electronics|909076000          |
|P05       |Router      |Networking |233766000          |
+----------+------------+-----------+-------------------+



Part C

In [11]:
exercise_9_df = sales_df.agg(
    F.sum(F.col("discount_pct").isNull().cast("long")).alias("null_discount_pct"),
    F.sum(F.col("salesperson_id").isNull().cast("long")).alias("null_salesperson_id"))
exercise_9_df.show()

+-----------------+-------------------+
|null_discount_pct|null_salesperson_id|
+-----------------+-------------------+
|             5263|               5882|
+-----------------+-------------------+



In [12]:
exercise_10_df = sales_df.na.fill({"discount_pct": 0})
exercise_10_df.select("order_id", "discount_pct").show(10)

+--------+------------+
|order_id|discount_pct|
+--------+------------+
|       3|           0|
|       9|           5|
|       2|           0|
|       8|           5|
|       4|           0|
|      10|          10|
|       5|           5|
|      11|          10|
|       1|           0|
|       7|           5|
+--------+------------+
only showing top 10 rows


In [13]:
exercise_11_df = sales_df.na.drop(subset=["salesperson_id"])
print("Rows before:", sales_df.count())
print("Rows after:", exercise_11_df.count())

Rows before: 100000
Rows after: 94118


In [14]:
exercise_12_df = sales_df.withColumn("order_size",
    F.when(F.col("quantity").between(1, 3), "Small")
     .when(F.col("quantity").between(4, 6), "Medium").otherwise("Large"))
exercise_12_df.select("order_id", "quantity", "order_size").show(12)

+--------+--------+----------+
|order_id|quantity|order_size|
+--------+--------+----------+
|       3|       4|    Medium|
|       9|       2|     Small|
|       2|       3|     Small|
|       8|       1|     Small|
|       4|       5|    Medium|
|      10|       3|     Small|
|       5|       6|    Medium|
|      11|       4|    Medium|
|       1|       2|     Small|
|       7|       8|     Large|
|      13|       6|    Medium|
|       6|       7|     Large|
+--------+--------+----------+
only showing top 12 rows


Part D

In [15]:
product_revenue_df = (sales_df.join(products_df, "product_id", "left")
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("product_id", "product_name", "category")
    .agg(F.sum("gross_revenue").alias("total_gross_revenue")))
category_window = Window.partitionBy("category").orderBy(F.desc("total_gross_revenue"))
exercise_13_df = product_revenue_df.withColumn("revenue_rank", F.dense_rank().over(category_window)).orderBy("category", "revenue_rank")
exercise_13_df.show(truncate=False)

+----------+------------+-----------+-------------------+------------+
|product_id|product_name|category   |total_gross_revenue|revenue_rank|
+----------+------------+-----------+-------------------+------------+
|P03       |Headphones  |Accessories|233334500          |1           |
|P01       |Laptop      |Electronics|4000080000         |1           |
|P06       |Tablet      |Electronics|1999920000         |2           |
|P02       |Monitor     |Electronics|1166704000         |3           |
|P05       |Router      |Networking |299992500          |1           |
|P04       |Printer     |Office     |1416678000         |1           |
+----------+------------+-----------+-------------------+------------+



In [16]:
store_running_window = Window.partitionBy("store_id").orderBy("order_date", "order_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
exercise_14_df = sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")).withColumn(
    "running_gross_revenue", F.sum("gross_revenue").over(store_running_window))
exercise_14_df.select("store_id", "order_date", "order_id", "gross_revenue", "running_gross_revenue").orderBy("store_id", "order_date", "order_id").show(20, truncate=False)

+--------+----------+--------+-------------+---------------------+
|store_id|order_date|order_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+-------------+---------------------+
|S01     |2026-01-01|180     |300000       |300000               |
|S01     |2026-01-01|360     |60000        |360000               |
|S01     |2026-01-01|540     |300000       |660000               |
|S01     |2026-01-01|720     |60000        |720000               |
|S01     |2026-01-01|900     |300000       |1020000              |
|S01     |2026-01-01|1080    |60000        |1080000              |
|S01     |2026-01-01|1260    |300000       |1380000              |
|S01     |2026-01-01|1440    |60000        |1440000              |
|S01     |2026-01-01|1620    |300000       |1740000              |
|S01     |2026-01-01|1800    |60000        |1800000              |
|S01     |2026-01-01|1980    |300000       |2100000              |
|S01     |2026-01-01|2160    |60000        |2160000           

In [17]:
exercise_15_df = (sales_df.filter(F.col("order_status") == "Completed")
    .withColumn("month", F.month("order_date"))
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .groupBy("month").agg(F.sum("gross_revenue").alias("monthly_sales_revenue")).orderBy("month"))
exercise_15_df.show()

+-----+---------------------+
|month|monthly_sales_revenue|
+-----+---------------------+
|    1|           1278728500|
|    2|           1027585500|
|    3|           1212087500|
|    4|           1218921000|
|    5|           1277560500|
|    6|           1089027500|
+-----+---------------------+



Part E

In [18]:
lazy_df_16 = sales_df.filter(F.col("order_status") == "Completed")
print("Transformation 1 defined: filter()")
lazy_df_16 = lazy_df_16.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("Transformation 2 defined: withColumn()")
lazy_df_16 = lazy_df_16.select("order_id", "product_id", "quantity", "gross_revenue")
print("Transformation 3 defined: select()")
print("show() triggers execution:")
lazy_df_16.show(10)

Transformation 1 defined: filter()
Transformation 2 defined: withColumn()
Transformation 3 defined: select()
show() triggers execution:
+--------+----------+--------+-------------+
|order_id|product_id|quantity|gross_revenue|
+--------+----------+--------+-------------+
|       3|       P04|       4|        68000|
|       9|       P04|       2|        34000|
|       2|       P03|       3|        10500|
|       8|       P03|       1|         3500|
|       4|       P05|       5|        22500|
|      10|       P05|       3|        13500|
|       5|       P06|       6|       144000|
|       1|       P02|       2|        28000|
|      13|       P02|       6|        84000|
|       6|       P01|       7|       420000|
+--------+----------+--------+-------------+
only showing top 10 rows


In [19]:
exercise_17_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity"))
exercise_17_df.explain()
collected_product_totals = exercise_17_df.collect()
print(collected_product_totals)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum(quantity#35)])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum(quantity#35)])
      +- Project [product_id#33, quantity#35]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=4575]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=4576]
                  +- Project [product_id#0]
                     +- Filter isnotnull(product_id#0)
                        +- Scan ExistingRDD

Part F

In [20]:
exercise_18_df = sales_df.filter(F.col("order_status") == "Completed").cache()
start = time.time()
count_first = exercise_18_df.count()
first_seconds = time.time() - start
start = time.time()
count_second = exercise_18_df.count()
second_seconds = time.time() - start
print(f"First count: {count_first:,}; elapsed: {first_seconds:.3f} seconds")
print(f"Second count: {count_second:,}; elapsed: {second_seconds:.3f} seconds")
exercise_18_df.unpersist()

First count: 77,923; elapsed: 6.970 seconds
Second count: 77,923; elapsed: 2.032 seconds


DataFrame[order_id: bigint, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: bigint, discount_pct: int, order_status: string, payment_method: string, salesperson_id: string]

In [21]:
exercise_19_df = sales_df.filter(F.col("quantity") > 5).persist(StorageLevel.MEMORY_AND_DISK)
print("Count:", exercise_19_df.count())
print("Sample rows:", len(exercise_19_df.take(10)))
exercise_19_df.unpersist()
print("Persisted DataFrame released.")

Count: 37500
Sample rows: 10
Persisted DataFrame released.


In [22]:
exercise_20_df = sales_df.groupBy("product_id").agg(F.sum("quantity").alias("total_quantity")).cache()
exercise_20_df.count()  # Materialize cache
exercise_20_df.explain(extended=True)
exercise_20_df.unpersist()

== Parsed Logical Plan ==
'Aggregate ['product_id], ['product_id, 'sum('quantity) AS total_quantity#1603]
+- Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
   +- Project [product_id#33, order_id#31L, order_date#32, store_id#34, quantity#35, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39, unit_price#40L]
      +- Join LeftOuter, (product_id#33 = product_id#0)
         :- Project [order_id#31L, order_date#32, product_id#33, store_id#34, quantity#35, discount_pct#36, order_status#37, payment_method#38, CASE WHEN ((order_id#31L % cast(17 as bigint)) = cast(0 as bigint)) THEN cast(null as string) ELSE concat(EMP, lpad(cast(((order_id#31L % cast(12 as bigint)) + cast(1 as bigint)) as string), 2, 0)) END AS salesperson_id#39]
         :  +- Project [order_id#31L, order_date#32, product_id#33, store_id#34, quantity#35, discount_pct#36, order_status#37,

DataFrame[product_id: string, total_quantity: bigint]

Part G

In [23]:
print("Existing partitions:", sales_df.rdd.getNumPartitions())
exercise_21_df = sales_df.repartition(8)
print("After repartition(8):", exercise_21_df.rdd.getNumPartitions())

Existing partitions: 2
After repartition(8): 8


In [24]:
exercise_23_df = sales_df.repartition(6, "store_id").withColumn("partition_id", F.spark_partition_id())
exercise_23_df.groupBy("partition_id").agg(
    F.collect_set("store_id").alias("stores_in_partition"), F.count("*").alias("record_count")
).orderBy("partition_id").show(truncate=False)

+------------+-------------------+------------+
|partition_id|stores_in_partition|record_count|
+------------+-------------------+------------+
|1           |[S03]              |24999       |
|3           |[S02]              |25001       |
|4           |[S01]              |25001       |
|5           |[S04]              |24999       |
+------------+-------------------+------------+



In [25]:
sales_12_partitions_df = sales_df.repartition(12)
coalesced_4_df = sales_12_partitions_df.coalesce(4)
repartitioned_4_df = sales_12_partitions_df.repartition(4)
print("12 partitions:", sales_12_partitions_df.rdd.getNumPartitions())
print("coalesce(4):", coalesced_4_df.rdd.getNumPartitions())
print("repartition(4):", repartitioned_4_df.rdd.getNumPartitions())
print("\nPlan for coalesce(4):")
coalesced_4_df.explain()
print("\nPlan for repartition(4):")
repartitioned_4_df.explain()

12 partitions: 12
coalesce(4): 4
repartition(4): 4

Plan for coalesce(4):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- Coalesce 4
      +- ShuffleQueryStage 3
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=6188]
            +- *(3) Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#40L, discount_pct#36, order_status#37, payment_method#38, salesperson_id#39]
               +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                  :- AQEShuffleRead local
                  :  +- ShuffleQueryStage 0
                  :     +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=6088]
                  :        +- *(1) Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, c

In [26]:
output_csv_path = "/content/sales_csv_output"
sales_df.repartition(4).write.mode("overwrite").option("header", True).csv(output_csv_path)
print("CSV output written to:", output_csv_path)

CSV output written to: /content/sales_csv_output


Final Challenge

In [29]:
completed_sales_df = sales_df.filter(F.col("order_status") == "Completed").na.fill({"discount_pct": 0})


In [30]:
challenge_joined_df = completed_sales_df.join(products_df, "product_id", "left").join(stores_df, "store_id", "left")


In [31]:
challenge_revenue_df = (challenge_joined_df
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
    .withColumn("discounted_revenue", F.col("gross_revenue") * (1 - F.col("discount_pct") / 100.0)))


In [32]:
challenge_aggregated_df = challenge_revenue_df.groupBy("region", "category").agg(
    F.sum("gross_revenue").alias("total_gross_revenue"),
    F.sum("discounted_revenue").alias("total_discounted_revenue"),
    F.count("*").alias("transaction_count"))


In [33]:
region_rank_window = Window.partitionBy("region").orderBy(F.desc("total_discounted_revenue"))
challenge_final_df = challenge_aggregated_df.withColumn("category_rank", F.dense_rank().over(region_rank_window)).cache()


In [34]:
challenge_final_df.orderBy("region", "category_rank").show(truncate=False)


+------+-----------+-------------------+------------------------+-----------------+-------------+
|region|category   |total_gross_revenue|total_discounted_revenue|transaction_count|category_rank|
+------+-----------+-------------------+------------------------+-----------------+-------------+
|North |Electronics|934848000          |8.551824E8              |6492             |1            |
|North |Office     |441388000          |4.12114E8               |6492             |2            |
|North |Networking |146097000          |1.35026775E8            |6494             |3            |
|South |Electronics|2156236000         |2.0242794E9             |19484            |1            |
|South |Office     |662524000          |6.060058E8              |6495             |2            |
|South |Accessories|113631000          |1.05018025E8            |6494             |3            |
|South |Networking |87669000           |8.2684125E7             |6494             |4            |
|West  |Electronics|

In [35]:
parquet_output_path = "/content/final_sales_summary_parquet"
challenge_final_df.repartition(2).write.mode("overwrite").parquet(parquet_output_path)
print("Parquet output written to:", parquet_output_path)


Parquet output written to: /content/final_sales_summary_parquet
